# Jacobian sensitivity

This notebook reproduces the surrogate, exact PyTheus, and absolute-difference Jacobian sensitivity heatmaps for $n=4,6,8$. The two sensitivity panels for each node count share a color scale so they can be compared directly.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_repository_root():
    """Support launching Jupyter from the repository root or plotting_code/."""
    current = Path.cwd().resolve()
    if (current / "ml_model.py").is_file() and (current / "plotting_code").is_dir():
        return current
    if current.name == "plotting_code" and (current.parent / "ml_model.py").is_file():
        return current.parent
    raise RuntimeError(
        "Launch this notebook from the PolySurrogate repository root "
        "or from its plotting_code directory."
    )


REPO_ROOT = find_repository_root()
PLOTTING_DIR = REPO_ROOT / "plotting_code"
DATA_DIR = PLOTTING_DIR / "Data" / "jacobian_data"
OUTPUT_DIR = PLOTTING_DIR / "Results" / "jacobian_sensitivity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NODE_COUNTS = (4, 6, 8)
TICK_STEPS = {
    4: {"x": 10, "y": 8},
    6: {"x": 10, "y": 16},
    8: {"x": 20, "y": 32},
}
SAVE_FORMATS = ("png", "pdf", "svg")
SAVE_DPI = 600
LABEL_FONTSIZE = 36
TICK_FONTSIZE = 34
COLORBAR_LABEL_FONTSIZE = 34
COLORBAR_TICK_FONTSIZE = 34

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 28,
    "font.weight": "bold",
    "axes.labelsize": 52,
    "axes.labelweight": "bold",
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "mathtext.fontset": "dejavuserif",
    "axes.linewidth": 1.8,
})

print("Jacobian data:", DATA_DIR)
print("Figure output:", OUTPUT_DIR)

## Load and validate sensitivity matrices

In [ ]:
MATRIX_FILES = {
    "nn": "mean_nn_sensitivity.npy",
    "pytheus": "mean_pytheus_sensitivity.npy",
    "difference": "difference_sensitivity.npy",
}


def load_matrices(node_count):
    node_dir = DATA_DIR / f"{node_count}_node"
    paths = {name: node_dir / filename for name, filename in MATRIX_FILES.items()}
    missing = [path for path in paths.values() if not path.is_file()]
    if missing:
        formatted = "\n".join(f"  - {path}" for path in missing)
        raise FileNotFoundError(f"Missing Jacobian matrices:\n{formatted}")

    matrices = {name: np.load(path) for name, path in paths.items()}
    expected_shape = (2**node_count, 2 * node_count * (node_count - 1))
    for name, matrix in matrices.items():
        if matrix.shape != expected_shape:
            raise ValueError(
                f"{paths[name]} has shape {matrix.shape}; expected {expected_shape}"
            )
        if not np.all(np.isfinite(matrix)):
            raise ValueError(f"Non-finite values found in {paths[name]}")

    # The difference was calculated before the mean matrices were saved as
    # float32, so allow the rounding error introduced by reloading them.
    expected_difference = np.abs(matrices["nn"] - matrices["pytheus"])
    if not np.allclose(
        matrices["difference"], expected_difference, rtol=1e-4, atol=1e-7,
    ):
        raise ValueError(f"Stored difference matrix is inconsistent for n={node_count}")
    return matrices


matrices_by_node = {node: load_matrices(node) for node in NODE_COUNTS}
for node_count, matrices in matrices_by_node.items():
    print(f"n={node_count}: matrix shape {matrices['nn'].shape}")

## Generate the heatmaps

In [ ]:
def style_axis(ax, matrix, node_count):
    steps = TICK_STEPS[node_count]
    x_ticks = np.arange(0, matrix.shape[1], steps["x"])
    y_ticks = np.arange(0, matrix.shape[0], steps["y"])
    ax.set_xticks(x_ticks)
    ax.set_yticks(y_ticks)
    ax.set_xticklabels(x_ticks)
    ax.set_yticklabels(y_ticks)
    ax.tick_params(
        axis="x", which="major", length=8, width=1.6, direction="out",
    )
    ax.tick_params(
        axis="y", which="major", left=True, right=False, length=8,
        width=1.6, direction="out", labelleft=True,
    )
    ax.set_xlabel(
        "Input graph-weight index", labelpad=18,
        fontsize=LABEL_FONTSIZE, fontweight="bold",
    )
    ax.set_ylabel(
        "Output amplitude index", labelpad=18,
        fontsize=LABEL_FONTSIZE, fontweight="bold",
    )
    ax.set_box_aspect(1)


def save_figure(fig, stem):
    for extension in SAVE_FORMATS:
        path = stem.with_suffix(f".{extension}")
        dpi = SAVE_DPI if extension == "png" else None
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        print(f"Saved: {path}")


def plot_matrix(node_count, panel_name, matrix, cmap, vmax, colorbar_label):
    node_output_dir = OUTPUT_DIR / f"{node_count}_node"
    node_output_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(14, 12))
    image = ax.imshow(
        matrix, origin="lower", interpolation="nearest", cmap=cmap,
        vmin=0.0, vmax=vmax, aspect="auto",
    )
    style_axis(ax, matrix, node_count)

    colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.06)
    colorbar.set_label(
        colorbar_label, fontsize=COLORBAR_LABEL_FONTSIZE,
        fontweight="bold", labelpad=20,
    )
    colorbar.ax.tick_params(
        labelsize=COLORBAR_TICK_FONTSIZE, width=1.6, length=7,
    )
    plt.setp(colorbar.ax.get_yticklabels(), fontweight="bold")
    fig.tight_layout()

    stem = node_output_dir / f"jacobian_sensitivity_{panel_name}_{node_count}_node"
    save_figure(fig, stem)
    plt.show()
    plt.close(fig)


for node_count, matrices in matrices_by_node.items():
    common_vmax = max(matrices["nn"].max(), matrices["pytheus"].max())
    plot_matrix(
        node_count, "nn_sensitivity", matrices["nn"], "viridis",
        common_vmax, "Mean Jacobian sensitivity",
    )
    plot_matrix(
        node_count, "pytheus_sensitivity", matrices["pytheus"], "viridis",
        common_vmax, "Mean Jacobian sensitivity",
    )
    plot_matrix(
        node_count, "difference", matrices["difference"], "magma",
        matrices["difference"].max(),
        r"$|S^{\mathrm{NN}}_{ij} - S^{\mathrm{PM}}_{ij}|$",
    )

print("\nGenerated all nine Jacobian sensitivity figures.")